In [64]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    RocCurveDisplay
)

import warnings
warnings.filterwarnings("ignore")

In [65]:
df=pd.read_csv("/Users/nidagallusandeep/Downloads/customer_purchase_20000.csv")

In [66]:
df.head()

,CustomerID,Age,Gender,AnnualIncome,SpendingScore,WebsiteVisits,PagesViewed,TimeOnWebsite,PreviousPurchases,DiscountUsed,Membership,ProductCategory,CartValue,Purchased
0,CUST00001,22,Female,23244.0,75.0,10,26,9.5,2,No,Silver,Beauty,141.67,0
1,CUST00002,55,Female,46391.0,91.0,5,13,10.0,2,No,Basic,Beauty,167.69,0
2,CUST00003,49,Male,43127.0,24.0,4,14,4.0,0,No,Silver,Home,167.11,0
3,CUST00004,39,Male,50254.0,47.0,9,25,6.7,7,Yes,Silver,Beauty,88.39,1
4,CUST00005,38,Male,71597.0,32.0,6,20,17.7,6,No,Basic,Sports,136.94,0


In [67]:
df.tail()

,CustomerID,Age,Gender,AnnualIncome,SpendingScore,WebsiteVisits,PagesViewed,TimeOnWebsite,PreviousPurchases,DiscountUsed,Membership,ProductCategory,CartValue,Purchased
19995,CUST19996,25,Female,77054.0,39.0,9,11,7.7,2,Yes,Platinum,Books,78.11,0
19996,CUST19997,25,Female,60448.0,43.0,5,21,6.9,3,No,Basic,Home,265.64,0
19997,CUST19998,35,Female,81085.0,40.0,10,18,6.5,2,No,Gold,Clothing,66.82,0
19998,CUST19999,35,Female,68363.0,85.0,13,34,10.8,0,Yes,Gold,Beauty,144.88,0
19999,CUST20000,41,Male,22189.0,54.0,1,2,11.0,4,No,Gold,Electronics,282.05,0


In [68]:
df.shape

(20000, 14)

In [69]:
df.columns

Index(['CustomerID', 'Age', 'Gender', 'AnnualIncome', 'SpendingScore',
       'WebsiteVisits', 'PagesViewed', 'TimeOnWebsite', 'PreviousPurchases',
       'DiscountUsed', 'Membership', 'ProductCategory', 'CartValue',
       'Purchased'],
      dtype='object')

In [70]:
df.size

280000

In [71]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CustomerID         20000 non-null  object 
 1   Age                20000 non-null  int64  
 2   Gender             20000 non-null  object 
 3   AnnualIncome       19722 non-null  float64
 4   SpendingScore      19805 non-null  float64
 5   WebsiteVisits      20000 non-null  int64  
 6   PagesViewed        20000 non-null  int64  
 7   TimeOnWebsite      19781 non-null  float64
 8   PreviousPurchases  20000 non-null  int64  
 9   DiscountUsed       20000 non-null  object 
 10  Membership         19751 non-null  object 
 11  ProductCategory    20000 non-null  object 
 12  CartValue          20000 non-null  float64
 13  Purchased          20000 non-null  int64  
dtypes: float64(4), int64(5), object(5)
memory usage: 2.1+ MB


In [72]:
df.describe()

,Age,AnnualIncome,SpendingScore,WebsiteVisits,PagesViewed,TimeOnWebsite,PreviousPurchases,CartValue,Purchased
count,20000.000000,19722.000000,19805.000000,20000.000000,20000.000000,19781.000000,20000.000000,20000.000000,20000.000000
mean,41.334500,64775.056333,50.744004,8.013200,18.801300,10.005485,2.989400,170.484071,0.241350
std,13.832313,21743.314560,28.897142,2.845071,7.669358,6.291197,1.724858,106.770690,0.427913
min,18.000000,15000.000000,1.000000,0.000000,1.000000,0.500000,0.000000,10.000000,0.000000
25%,29.000000,49948.250000,26.000000,6.000000,13.000000,5.400000,2.000000,102.497500,0.000000
50%,41.000000,64608.500000,51.000000,8.000000,18.000000,8.800000,3.000000,146.950000,0.000000
75%,53.000000,79443.500000,76.000000,10.000000,23.000000,13.300000,4.000000,209.300000,0.000000
max,65.000000,175159.000000,100.000000,23.000000,67.000000,54.400000,12.000000,1316.620000,1.000000


In [73]:
df.isnull().sum()

CustomerID             0
Age                    0
Gender                 0
AnnualIncome         278
SpendingScore        195
WebsiteVisits          0
PagesViewed            0
TimeOnWebsite        219
PreviousPurchases      0
DiscountUsed           0
Membership           249
ProductCategory        0
CartValue              0
Purchased              0
dtype: int64

In [74]:
df["AnnualIncome"]=df["AnnualIncome"].fillna(df["AnnualIncome"].mean())
df["SpendingScore"]=df["SpendingScore"].fillna(df["SpendingScore"].mean())
df["TimeOnWebsite"]=df["TimeOnWebsite"].fillna(df["TimeOnWebsite"].mean())
df["Membership"]=df["Membership"].fillna(df["Membership"].mode()[0])

In [75]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CustomerID         20000 non-null  object 
 1   Age                20000 non-null  int64  
 2   Gender             20000 non-null  object 
 3   AnnualIncome       20000 non-null  float64
 4   SpendingScore      20000 non-null  float64
 5   WebsiteVisits      20000 non-null  int64  
 6   PagesViewed        20000 non-null  int64  
 7   TimeOnWebsite      20000 non-null  float64
 8   PreviousPurchases  20000 non-null  int64  
 9   DiscountUsed       20000 non-null  object 
 10  Membership         20000 non-null  object 
 11  ProductCategory    20000 non-null  object 
 12  CartValue          20000 non-null  float64
 13  Purchased          20000 non-null  int64  
dtypes: float64(4), int64(5), object(5)
memory usage: 2.1+ MB


In [76]:
df.isnull().sum()

CustomerID           0
Age                  0
Gender               0
AnnualIncome         0
SpendingScore        0
WebsiteVisits        0
PagesViewed          0
TimeOnWebsite        0
PreviousPurchases    0
DiscountUsed         0
Membership           0
ProductCategory      0
CartValue            0
Purchased            0
dtype: int64

In [77]:
df.duplicated().sum()

np.int64(0)

In [78]:
df = df.drop(columns=["CustomerID"])

In [79]:
X=df.drop(columns="Purchased",axis=1)
y=df["Purchased"]

# Identify numerical and categorical columns

In [80]:
num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
print("Numerical Columns:", num_cols)
print("Categorical Columns:", cat_cols)

Numerical Columns: ['Age', 'AnnualIncome', 'SpendingScore', 'WebsiteVisits', 'PagesViewed', 'TimeOnWebsite', 'PreviousPurchases', 'CartValue']
Categorical Columns: ['Gender', 'DiscountUsed', 'Membership', 'ProductCategory']


In [81]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

In [82]:
numerical_transformer=Pipeline(steps=[
    ("impute",SimpleImputer(strategy="median")),
    ("scaler",StandardScaler())
])

categorical_transformer=Pipeline(steps=[
    ("impute",SimpleImputer(strategy="most_frequent")),
    ("encoder",OneHotEncoder(handle_unknown="ignore"))
])

In [83]:
preprocessing=ColumnTransformer(transformers=[
    ("num",numerical_transformer,num_cols),
    ("cat",categorical_transformer,cat_cols)
])

# Train-test split

In [84]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)
print(X_train.shape)
print(X_test.shape) 
print(y_train.shape)
print(y_test.shape)

(16000, 12)
(4000, 12)
(16000,)
(4000,)


# Build Random Forest Classifier

In [85]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

rf_model = RandomForestClassifier(
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

model = Pipeline([
    ("preprocessing", preprocessing),
    ("model", rf_model)
])
model.fit(X_train,y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessing', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](12,)","['Age','Gender','AnnualIncome',...,'Membership','ProductCategory', 'CartValue']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,12
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough

In [91]:
y_pred=model.predict(X_test)

In [93]:
from sklearn.metrics import accuracy_score
accuracy=accuracy_score(y_test,y_pred)
print(accuracy)

0.7455


In [95]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, average="weighted", zero_division=0))
print("Recall:", recall_score(y_test, y_pred, average="weighted", zero_division=0))
print("F1 Score:", f1_score(y_test, y_pred, average="weighted", zero_division=0))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, zero_division=0))

Accuracy: 0.7455
Precision: 0.7305409126316134
Recall: 0.7455
F1 Score: 0.7365564650853206

Classification Report:

              precision    recall  f1-score   support

           0       0.81      0.86      0.84      3035
           1       0.47      0.38      0.42       965

    accuracy                           0.75      4000
   macro avg       0.64      0.62      0.63      4000
weighted avg       0.73      0.75      0.74      4000

